# Lesson 4: How to Bring Your Transformers Model to vLLM

In Lesson 3, we saw that the Transformers backend can load any *compatible* Hugging Face model in vLLM. But what makes a model compatible?

In this lesson, we'll reverse-engineer **OLMoE** to understand exactly what's needed. We'll walk through each requirement in the actual source code.

By the end, you'll have a clear checklist for making any Transformers model work with vLLM. For the canonical upstream reference, see Hugging Face's [Transformers as a backend](https://huggingface.co/docs/transformers/en/community_integrations/transformers_as_backend) guide.

## The Compatibility Checklist

A Transformers model needs these things to work with vLLM's Transformers backend:

1. `ALL_ATTENTION_FUNCTIONS` dispatch in the attention forward method and `layer_idx` in the attention module attributes
2. `_supports_attention_backend = True` on the `PreTrainedModel` (indicates that you have done step 1)
3. `**kwargs` pass-through of the forward chain from base model to attention interface call
4. **(MoE models only)** A replaceable `experts` submodule with a separate router, so `MoEMixin` can swap it for vLLM's `FusedMoE`
5. `base_model_tp_plan` in the config (tensor parallelism)
6. `base_model_pp_plan` in the config (pipeline parallelism)

Let's examine each one in OLMoE's code.

## Step 1: The Attention Interface — `ALL_ATTENTION_FUNCTIONS`

Lesson 3 covered the vLLM side of the bridge: the backend registers `vllm_flash_attention_forward` under `"vllm"` in `ALL_ATTENTION_FUNCTIONS`, and `_patch_config()` sets the model's `_attn_implementation` to `"vllm"` so that key gets picked up. From the model's side, that only works if its attention forward dispatches through the registry instead of hardcoding the computation:

```python
attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
    self.config._attn_implementation, eager_attention_forward
)
attn_output, attn_weights = attention_interface(
    self, query, key, value, attention_mask, **kwargs
)
```

`ALL_ATTENTION_FUNCTIONS` behaves like a dict, so the equivalent subscript form also works:

```python
attention_interface = ALL_ATTENTION_FUNCTIONS[self.config._attn_implementation]
```

The `get_interface(...)` helper is just a thin wrapper that adds extra validation (checking the key exists, falling back to a default like `eager_attention_forward`, etc.) — the actual lookup is a plain mapping access.

Let's look at OLMoE's attention forward:

In [ ]:
import inspect
from transformers.models.olmoe.modular_olmoe import OlmoeAttention

source = inspect.getsource(OlmoeAttention.__init__)
print(source)
source = inspect.getsource(OlmoeAttention.forward)
print(source)

Notice the key elements:
1. The signature includes `**kwargs: Unpack[TransformersKwargs]`
2. It calls `ALL_ATTENTION_FUNCTIONS.get_interface(self.config._attn_implementation, ...)` to get the attention function
3. It passes `**kwargs` to the attention interface

## Step 2: The Gate — `_supports_attention_backend`

This class attribute confirms that the model's attention forward dispatches through `ALL_ATTENTION_FUNCTIONS`. If it's `False` (or missing), the model is invisible to the Transformers backend — vLLM checks this via the `is_backend_compatible()` method when resolving which model class to use.

In [ ]:
from transformers import OlmoeModel

print(f"OlmoeModel._supports_attention_backend = {OlmoeModel._supports_attention_backend}")

Let's see where OLMoE sets this flag:

In [ ]:
import inspect
from transformers.models.olmoe.modular_olmoe import OlmoePreTrainedModel

source = inspect.getsource(OlmoePreTrainedModel)
print(source)

## Step 3: The `**kwargs` Chain

Lesson 3 covered how `attention_instances` rides the `**kwargs` chain from `Base.forward` down to the registered attention function. For that to work, every `forward` in between has to accept and forward `**kwargs` — drop it at any level and the attention instances are lost.

Let's verify that OLMoE passes `**kwargs` at every level:

In [ ]:
import inspect
from transformers.models.olmoe.modular_olmoe import (
    OlmoeModel,
    OlmoeDecoderLayer,
    OlmoeAttention,
)

classes = [
    ("OlmoeModel", OlmoeModel),
    ("OlmoeDecoderLayer", OlmoeDecoderLayer),
    ("OlmoeAttention", OlmoeAttention),
]

for name, cls in classes:
    sig = inspect.signature(cls.forward)
    params = list(sig.parameters.keys())
    has_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD
        for p in sig.parameters.values()
    )
    marker = "has **kwargs" if has_kwargs else "MISSING **kwargs!"
    print(f"{name}.forward({', '.join(params)}) -> {marker}")

All three levels forward `**kwargs`, so `attention_instances` has an unbroken path from `Base.forward` down to the attention function.

## Step 4 (MoE only): Replaceable Experts

Lesson 3 covered how `MoEMixin.recursive_replace()` swaps the Transformers `experts` submodule for vLLM's `FusedMoE`. The motivation isn't really about fusing N tiny expert MLPs — Transformers v5 already packs them into a grouped-MM kernel. The win is that `FusedMoE` plugs the MoE layer into vLLM's serving features: **expert parallelism (EP)**, **expert-parallel load balancing (EPLB)**, and vLLM's quantization and parallel-init machinery. Stay on the Transformers expert module and those features aren't available on this layer.

Three properties make this MoE block replaceable:

1. **The experts live in a submodule named `experts`.** `MoEMixin.recursive_replace()` walks the model tree looking specifically for modules with that attribute name — if it were called `moe_experts` or `mlp`, the backend would skip it.
2. **The router is a *sibling* of the experts, and `experts.forward` receives `(hidden_states, top_k_index, top_k_weights)`.** `self.gate` runs first and produces `(top_k_weights, top_k_index)`; those are then passed into `self.experts(hidden_states, top_k_index, top_k_weights)`. That signature is what makes the swap viable: once `FusedMoE` replaces `experts`, it receives the routing decisions Transformers already made as positional arguments and intercepts them via a `custom_routing_function` instead of recomputing them. If the experts module took a different signature (e.g. only `hidden_states` with routing fused inside), `FusedMoE` would have nothing to intercept.
3. **Expert weights are packed 3D tensors or `nn.ModuleList`.** `OlmoeExperts` inherits `MixtralExperts`, which stores weights as `gate_up_proj` of shape `(num_experts, 2 * intermediate_dim, hidden_dim)` and `down_proj` of shape `(num_experts, hidden_dim, intermediate_dim)`. vLLM also knows how to load experts that were saved in the old `nn.ModuleList` format.

Let's look at OLMoE's MoE block:

In [ ]:
import inspect
from transformers.models.olmoe.modeling_olmoe import OlmoeExperts, OlmoeSparseMoeBlock

print(inspect.getsource(OlmoeSparseMoeBlock))
print(f"class OlmoeExperts(nn.Module):\n{inspect.getsource(OlmoeExperts.__init__)}")

## Step 5: Tensor Parallel Plan

The `base_model_tp_plan` in the config tells vLLM (and PyTorch's tensor parallelism) how to shard each linear layer across GPUs:

- **`"colwise"`** — column-parallel: each GPU gets a slice of the output dimension. Used for Q, K, V, gate, and up projections.
- **`"rowwise"`** — row-parallel: each GPU gets a slice of the input dimension. Used for O and down projections.
- **`"packed_colwise"`** — column-parallel for fused/packed weight matrices.

OLMoE has some special entries due to its Q/K norm:

In [ ]:
from transformers import OlmoeConfig

print("OlmoeConfig.base_model_tp_plan:")
for pattern, style in OlmoeConfig.base_model_tp_plan.items():
    print(f"  {pattern:50s} -> {style}")

Notice the `"colwise_gather_output"` for Q, K, V projections and `"rowwise_split_input"` for O projections. These variants are needed because OLMoE applies RMSNorm to Q and K *after* projection — the norm needs the full (gathered) output, so the standard column-parallel approach (which keeps outputs sharded) doesn't work directly.

You'll also see MoE expert entries in the plan (`"moe_tp_experts"` for the expert module, `"packed_colwise"` / `"rowwise"` for the individual expert weights) — but the Transformers backend **doesn't actually use those** for sharding. `MoEMixin.recursive_replace()` has already swapped the `experts` module for vLLM's `FusedMoE` by the time TP is applied, and `FusedMoE` handles its own tensor-parallel sharding internally. The plan entries are still useful when the native Transformers model is run with PyTorch's TP (e.g. for training), just not on the vLLM path.

## Step 6: Pipeline Parallel Plan

The `base_model_pp_plan` defines how to split the model across pipeline stages. It maps module names to their position in the pipeline. Unlike TP (which splits layers), PP splits the model into sequential stages.

In [ ]:
print("OlmoeConfig.base_model_pp_plan:", OlmoeConfig.base_model_pp_plan)

In [ ]:
from transformers import LlamaConfig

print("LlamaConfig.base_model_pp_plan:")
for module, (inputs, outputs) in LlamaConfig.base_model_pp_plan.items():
    print(f"  {module:20s} inputs={inputs}, outputs={outputs}")

## Verifying It Works

Let's verify the full path: from model registry resolution to actual generation.

In [ ]:
from vllm.model_executor.models.registry import _VLLM_MODELS

# Show how vLLM resolves OLMoE's architecture
from transformers import AutoConfig

config = AutoConfig.from_pretrained("allenai/OLMoE-1B-7B-0125-Instruct")
arch = config.architectures[0]
print(f"Model architectures: {config.architectures}")
print(f"  -> '{arch}' is in the native registry: {arch in _VLLM_MODELS}")
print(f"  -> So vLLM uses the native implementation by default")
print(f"  -> With model_impl='transformers', it uses TransformersMoEForCausalLM instead")

In [ ]:
import os
from vllm import LLM, SamplingParams

MODEL_NAME = "allenai/OLMoE-1B-7B-0125-Instruct"

os.environ["VLLM_CPU_KVCACHE_SPACE"] = "1"
llm = LLM(
    model=MODEL_NAME,
    max_model_len=4096,
    enforce_eager=True,
    model_impl="transformers",
)

outputs = llm.generate(
    ["What makes mixture-of-experts models efficient?"],
    SamplingParams(max_tokens=100, temperature=0.7),
)

print(outputs[0].outputs[0].text)

## Common Pitfalls

When making a model compatible with the Transformers backend, watch out for:

1. **Missing `**kwargs` at any level** — The most common issue. If `OlmoeModel.forward` accepted `**kwargs` but `OlmoeDecoderLayer.forward` didn't, `attention_instances` would be silently dropped.

2. **Custom attention not using `ALL_ATTENTION_FUNCTIONS`** — Models that compute attention inline can't be dispatched to vLLM's kernels. The model must use the standard dispatch pattern.

3. **Incorrect TP plans** — Misspecifying `"colwise"` vs `"rowwise"` will produce wrong results silently. Remember: projections that *increase* dimension (Q, K, V, gate, up) are typically `"colwise"`, and projections that *decrease* dimension (O, down) are `"rowwise"`.

4. **Non-standard attention mask handling** — Models that manipulate attention weights directly (e.g., adding positional bias to attention scores after softmax) may not be expressible through the standard attention interface.

## Summary

To make a Transformers model work with vLLM's Transformers backend:

| # | Requirement | Where | Why |
|---|-------------|-------|-----|
| 1 | `ALL_ATTENTION_FUNCTIONS` dispatch | Attention `.forward()` | Allows vLLM to inject its attention kernels |
| 2 | `_supports_attention_backend = True` | `PreTrainedModel` class | Gate check for backend compatibility |
| 3 | `**kwargs` pass-through | Every `.forward()` from base model to attention interface call | Routes `attention_instances` to the attention interface |
| 4 | Replaceable `experts` submodule with sibling router (MoE only) | MoE block | Lets `MoEMixin` swap the experts for vLLM's `FusedMoE` |
| 5 | `base_model_tp_plan` | Config class | Tells vLLM how to shard layers for tensor parallelism |
| 6 | `base_model_pp_plan` | Config class | Tells vLLM how to split the model for pipeline parallelism |

Contributing these changes upstream to Transformers benefits the entire ecosystem — once a model is compatible, it works in vLLM, SGLang, and any other engine that uses the Transformers backend.